# CheMeleon GNN Fingerprints

This notebook demonstrates how to generate CheMeleon molecular embeddings from SMILES using Mother.

In [ ]:
import os
from pathlib import Path

import numpy as np
from rdkit import Chem

from mother.feature_generation.fp_gnn_gen import CheMeleonFingerprintFactory

In [ ]:
smiles = ["CCO", "c1ccccc1", "CC(=O)O"]
mols = [Chem.MolFromSmiles(s) for s in smiles]
mols_array = np.array(mols, dtype=object)

checkpoint_path = os.environ.get("CHEMELEON_CHECKPOINT_PATH", "").strip()
if not checkpoint_path:
    raise ValueError("Set CHEMELEON_CHECKPOINT_PATH to a local CheMeleon/Chemprop checkpoint.")

ckpt = Path(checkpoint_path).expanduser().resolve()
if not ckpt.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt}")

factory = CheMeleonFingerprintFactory(output_dim=2048, batch_size=128, checkpoint_path=str(ckpt), device="cpu")
transformer = factory.get_fingerprint_generator()
embeddings = transformer.fit_transform(mols_array)

print("Embedding matrix shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)
print("First row (first 8 values):", np.round(embeddings[0][:8], 4))

## Notes

- Install optional dependency first: `pip install "mother[gnn]"`.
- Set a checkpoint path before running: `export CHEMELEON_CHECKPOINT_PATH=/absolute/path/to/chemeleon.ckpt`.
- Input to the transformer should be RDKit molecule objects, matching other fingerprint generators.
- Invalid molecules are returned as rows with `NaN` values.